# Maia Corpus v9 — Acquisition & Audit v0.2 (Google Drive persistent)

All **persistent inputs and outputs** are stored in Google Drive under `MyDrive/Colab_LLMs/Maia_Lite_0_4_0/maia-corpus-v9/`. The notebook mounts Drive and fails closed if unavailable. FineMath remains excluded. This is a pilot auditor, **not an automatic dataset downloader**.


In [ ]:
# 1 — Dependencies (minimal; no torch/transformers modifications)
%pip -q install 'datasets>=3,<5' 'tqdm>=4.66' 'pandas>=2' 'pyarrow>=15' 'requests>=2.31'

In [ ]:
# 2 — Persistent Google Drive configuration
from pathlib import Path
import os

# All persistent input data, accepted records, quarantine and reports go to Drive.
DRIVE_ROOT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_4_0/maia-corpus-v9')
PROJECT_DIR = DRIVE_ROOT / 'pilot'
INPUTS = {
    'math': [str(DRIVE_ROOT / 'inputs' / 'math')],
    'code': [str(DRIVE_ROOT / 'inputs' / 'code')],
    'books': [str(DRIVE_ROOT / 'inputs' / 'books')],
}
MAX_SCAN_PER_DOMAIN = 10_000
MAX_ACCEPT_PER_DOMAIN = 2_000
MAX_CHARS_PER_DOCUMENT = 150_000
MIN_CHARS = {'math': 200, 'code': 100, 'books': 500}
ALLOWED_LICENSES = {
    'CC0-1.0', 'CC-BY-4.0', 'CC-BY-SA-4.0',
    'MIT', 'Apache-2.0', 'BSD-2-Clause', 'BSD-3-Clause',
    'Unlicense', '0BSD',
}
SEED = 20261122


In [ ]:
# 3 — Mount Drive and fail closed if persistence is unavailable
from google.colab import drive
drive.mount('/content/drive')
assert Path('/content/drive/MyDrive').is_dir(), 'Google Drive not mounted: refusing to write to ephemeral storage.'
root = Path(PROJECT_DIR)
assert root.is_relative_to(Path('/content/drive/MyDrive')), 'Output must be inside Google Drive.'
for name in ['raw_manifest', 'accepted', 'quarantine', 'reports']:
    (root / name).mkdir(parents=True, exist_ok=True)
for paths in INPUTS.values():
    for p in paths:
        target = Path(p)
        assert target.is_relative_to(Path('/content/drive/MyDrive')), 'Input must be on Google Drive.'
        target.mkdir(parents=True, exist_ok=True)
print('Persistent workspace:', root)
print('Persistent input folders:', INPUTS)
print('Note: files already written to /content by earlier runs are NOT moved automatically.')


## Source registry and licensing

These are **candidates**, not approved datasets. For arXiv, the license must be checked **per paper**. For Gutenberg, public-domain status can vary by jurisdiction. For code, a repository license must be verified at the relevant snapshot/file level, including attribution requirements. Mathematical text may contain extracted or machine-generated markup requiring quality review.

In [ ]:
# 4 — Candidate registry; status is intentionally NOT approved
import json, datetime
registry = [
 {'source':'OpenWebMath','domain':'math','status':'review','risks':'web provenance, content quality, licenses, duplication'},
 {'source':'arXiv source LaTeX','domain':'math','status':'review','risks':'per-paper licensing, TeX macros, figures, attribution'},
 {'source':'OpenStax','domain':'math','status':'review','risks':'edition/license/attribution, structured extraction'},
 {'source':'Proof-Pile-2','domain':'math','status':'review','risks':'component-level provenance, terms, overlap'},
 {'source':'The Stack / BigCode','domain':'code','status':'review','risks':'file/repository licensing, opt-outs, secrets, duplicates'},
 {'source':'Software Heritage','domain':'code','status':'review','risks':'repository licensing, personal data, provenance'},
 {'source':'Project Gutenberg','domain':'books','status':'review','risks':'jurisdiction, header/footer removal, editions'},
 {'source':'FineMath','domain':'math','status':'excluded','risks':'previous issues; do not ingest'},
]
(root/'raw_manifest'/'candidate_sources.json').write_text(json.dumps(registry,indent=2,ensure_ascii=False),encoding='utf-8')
import pandas as pd
display(pd.DataFrame(registry))

## Expected local record format

Recommended JSONL record (one JSON object per line):

```json
{"text":"...", "source":"OpenStax", "source_id":"book/chapter/section", "url":"https://...", "license":"CC-BY-4.0", "license_evidence":"https://...", "language":"en", "title":"..."}
```

The `license` and `license_evidence` fields are required for acceptance. This notebook checks metadata presence and allowlist membership, **not the legal validity of the evidence**. For `.txt` files, no license is inferred: they enter quarantine until metadata is supplied. Do not place confidential code or secrets in the input folders.

In [ ]:
# 5 — Streaming readers (local files only; bounded pilot)
import gzip, hashlib, re, itertools, collections, random
from pathlib import Path

def open_text(path):
    return gzip.open(path,'rt',encoding='utf-8',errors='replace') if path.name.endswith('.gz') else open(path,'r',encoding='utf-8',errors='replace')

def iter_records(domain):
    for folder in INPUTS[domain]:
        for path in sorted(Path(folder).rglob('*')):
            if not path.is_file() or not (path.name.endswith(('.jsonl','.jsonl.gz','.txt','.txt.gz'))):
                continue
            if path.name.endswith(('.txt','.txt.gz')):
                with open_text(path) as f:
                    text = f.read(MAX_CHARS_PER_DOCUMENT + 1)
                yield {'text':text,'source':'local-file','source_id':str(path),'url':'','license':'','license_evidence':'','language':'unknown'}, str(path)
            else:
                with open_text(path) as f:
                    for line_no, line in enumerate(f,1):
                        try:
                            obj = json.loads(line)
                            if isinstance(obj,dict):
                                yield obj, f'{path}:{line_no}'
                        except json.JSONDecodeError:
                            yield {'text':'','_parse_error':'invalid JSON'}, f'{path}:{line_no}'

In [ ]:
# 6 — Conservative filtering and SHA-256 exact deduplication
import unicodedata

def clean_text(s):
    # Keep mathematical symbols, indentation, code whitespace, and Unicode unchanged.
    return s.replace('\x00','')

def license_ok(r):
    return (r.get('license') in ALLOWED_LICENSES and
            isinstance(r.get('license_evidence'),str) and
            r['license_evidence'].startswith(('https://','http://')))

def review_record(r, domain, seen):
    if r.get('_parse_error'): return None, 'invalid_json'
    raw = r.get('text','')
    if not isinstance(raw,str): return None, 'missing_text'
    if len(raw) > MAX_CHARS_PER_DOCUMENT: return None, 'oversized_document'
    txt = clean_text(raw)
    if len(txt.strip()) < MIN_CHARS[domain]: return None, 'too_short'
    if not r.get('source') or not r.get('source_id'): return None, 'missing_provenance'
    if not license_ok(r): return None, 'license_unverified'
    digest = hashlib.sha256(txt.encode('utf-8')).hexdigest()
    if digest in seen: return None, 'exact_duplicate'
    seen.add(digest)
    return {'text':txt,'sha256':digest,'domain':domain,
            'source':r['source'],'source_id':str(r['source_id']),
            'url':r.get('url',''),'license':r['license'],
            'license_evidence':r['license_evidence'],
            'language':r.get('language','unknown'),'title':r.get('title','')}, None

In [ ]:
# 7 — Pilot audit and quarantine: stores metadata only for rejected records
from tqdm.auto import tqdm
seen = set()
summary = []
for domain in INPUTS:
    reasons = collections.Counter()
    accepted = 0
    examined = 0
    output = root/'accepted'/f'{domain}.jsonl'
    rejected = root/'quarantine'/f'{domain}_reasons.jsonl'
    with output.open('w',encoding='utf-8') as fout, rejected.open('w',encoding='utf-8') as frej:
        for r, location in itertools.islice(iter_records(domain), MAX_SCAN_PER_DOMAIN):
            examined += 1
            item, reason = review_record(r,domain,seen)
            if reason:
                reasons[reason] += 1
                frej.write(json.dumps({'location':location,'reason':reason,'source':r.get('source',''), 'source_id':r.get('source_id','')},ensure_ascii=False)+'\n')
                continue
            fout.write(json.dumps(item,ensure_ascii=False)+'\n')
            accepted += 1
            if accepted >= MAX_ACCEPT_PER_DOMAIN: break
    summary.append({'domain':domain,'examined':examined,'accepted':accepted,'rejected':examined-accepted,'reasons':dict(reasons)})
    print(domain, 'examined=',examined,'accepted=',accepted,'reasons=',dict(reasons))
(root/'reports'/'pilot_summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
display(pd.DataFrame(summary))

## Token estimates and content checks

The estimate below uses a rough **character-based proxy**. It is **not** the final token count. Once MaiaTokenizer-65k is trained, re-run the count with that exact tokenizer. Do not compare budgets across incompatible tokenizers as though they were identical.

In [ ]:
# 8 — Metrics by domain and source
stats = []
for domain in INPUTS:
    path = root/'accepted'/f'{domain}.jsonl'
    with path.open(encoding='utf-8') as f:
        for line in f:
            r = json.loads(line)
            t = r['text']
            stats.append({'domain':domain,'source':r['source'],'license':r['license'],
                          'language':r.get('language','unknown'),'chars':len(t),
                          'approx_tokens_chars_div_4':round(len(t)/4),
                          'lines':t.count('\n')+1,
                          'has_latex':bool(re.search(r'\\(?:begin|frac|sum|int|alpha|beta)\b|\$[^\n]+\$',t)),
                          'has_code_markers':bool(re.search(r'(?m)^\s*(def |class |#include|function |import |public class )',t))})
df = pd.DataFrame(stats)
if df.empty:
    print('No accepted records. This is expected until you provide JSONL with verified provenance/license metadata.')
else:
    display(df.groupby(['domain','source'],dropna=False).agg(documents=('chars','size'),chars=('chars','sum'),approx_tokens=('approx_tokens_chars_div_4','sum'),latex_docs=('has_latex','sum'),code_docs=('has_code_markers','sum')).reset_index())
    print('NOTE: approx_tokens is only a proxy, not MaiaTokenizer token counts.')

In [ ]:
# 9 — Deterministic manual review sample; inspect quality before scaling
rng = random.Random(SEED)
for domain in INPUTS:
    p = root/'accepted'/f'{domain}.jsonl'
    with p.open(encoding='utf-8') as f:
        records = [json.loads(line) for line in itertools.islice(f,MAX_ACCEPT_PER_DOMAIN)]
    picks = rng.sample(records,min(3,len(records)))
    print('\n###',domain,'sample count:',len(picks))
    for r in picks:
        print('SOURCE:',r['source'],'LICENSE:',r['license'],'ID:',r['source_id'])
        print(r['text'][:800].replace('\x00',''))
        print('-'*60)

In [ ]:
# 10 — Reproducibility manifest: file hashes and run configuration
import sys, platform
manifest = {
 'project':'Maia Corpus v9','phase':'pilot audit','version':'0.2',
 'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'python':sys.version,'platform':platform.platform(),
 'seed':SEED,'max_scan_per_domain':MAX_SCAN_PER_DOMAIN,
 'max_accept_per_domain':MAX_ACCEPT_PER_DOMAIN,
 'allowed_license_identifiers':sorted(ALLOWED_LICENSES),
 'fine_math_excluded':True,'files':{},'summary':summary,
 'limitations':['metadata allowlist is not legal review','exact dedup only','no cross-corpus v8 dedup','approx token counts are not tokenizer counts']
}
for path in sorted(root.rglob('*')):
    if path.is_file() and path.name != 'run_manifest.json':
        h=hashlib.sha256()
        with path.open('rb') as f:
            for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
        manifest['files'][str(path.relative_to(root))]={'sha256':h.hexdigest(),'bytes':path.stat().st_size}
manifest_path = root/'reports'/'run_manifest.json'
manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print('Saved:',manifest_path)
print('Next: approve specific sources and implement source-specific, license-aware adapters.')

## Acceptance criteria before large-scale acquisition

1. Verify **source and document-level licenses** and retain evidence/attribution. Review privacy, opt-outs and source-specific usage restrictions.
2. Inspect samples for corrupted math/LaTeX, boilerplate, low-quality code, secrets, and multilingual coverage.
3. Implement scalable exact **and near-duplicate** detection, including cross-domain and v8 overlap; current pilot only performs exact dedup within the accepted pilot.
4. Train and freeze **MaiaTokenizer-65k** on representative raw texts, validate HF/GGUF interoperability, then recount and tokenize the full corpus.
5. Reconcile final proportions using actual tokenizer counts. Preserve v8 snapshots and hashes; never overwrite them.

**Suggested next implementation:** a source-specific adapter for one verified OpenStax book and one verified code dataset partition, with resumable downloads and provenance-preserving extraction. The notebook intentionally does not download unreviewed datasets.